<a href="https://colab.research.google.com/github/chanu-24/self-driving-Car-Vision-/blob/main/ROI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset

class NuScenesGeometryDataset(Dataset):
    def __init__(self, img_shape=(480, 640), eps=1e-5):
        """
        nuScenes 데이터셋 구조를 모사하여 기하학적 3D-to-2D 투영 파이프라인을 일괄 관리하는 데이터셋 클래스입니다.
        Args:
            img_shape (tuple): (Height, Width) - 카메라 해상도 규격
            eps (float): 분모가 0이 되거나 음수가 되는 것을 방지하기 위한 소수 임계값
        """
        self.H, self.W = img_shape
        self.eps = eps

    def get_valid_mask(self, pixels_2d, points_cam):
        """
        [Day 21 핵심 연산 수식 구현]
        이미지 바운더리 조건 (0 <= u < W) & (0 <= v < H) 및 전방 깊이 조건 (Z > 0)을 체크하여 마스크를 생성합니다.

        Args:
            pixels_2d (Tensor): [B, Num_Cam, N, 2] (u, v 픽셀 좌표)
            points_cam (Tensor): [B, Num_Cam, N, 3] (X, Y, Z 카메라 3D 좌표)
        Returns:
            valid_mask (Tensor): [B, Num_Cam, N] (True/False 부울 마스크)
        """
        # 1. 픽셀 좌표 u, v 추출
        u = pixels_2d[..., 0] # [B, Num_Cam, N]
        v = pixels_2d[..., 1] # [B, Num_Cam, N]

        # 2. 카메라 기준 Z축 실제 거리(깊이) 추출
        Z = points_cam[..., 2] # [B, Num_Cam, N]

        # 3. [바운더리 수학 수식 반영]: 세 조건의 융합 교집합 연산을 부울 연산자(&)로 처리합니다.
        mask_u = (u >= 0) & (u < self.W)
        mask_v = (v >= 0) & (v < self.H)
        mask_z = Z > self.eps

        # 4. 최종 유효한 포인트 마스크 융합 (모두 만족해야 True)
        valid_mask = mask_u & mask_v & mask_z # [B, Num_Cam, N]

        return valid_mask

    def pipeline_transform(self, points_ego, rot_ego_to_cam, trans_ego_to_cam, intrinsics):
        """
        [Day 19 + Day 20 + Day 21 융합]: 수식 연산을 일괄적으로 관장하는 가속 메소드
        """
        B, N, _ = points_ego.shape
        _, Num_Cam, _, _ = rot_ego_to_cam.shape

        # ==========================================
        # [Day 19]: Extrinsic Matrix (외재 재원) 연산
        # ==========================================
        points_ego_expanded = points_ego.unsqueeze(1).expand(B, Num_Cam, N, 3)
        trans_expanded = trans_ego_to_cam.unsqueeze(2)
        points_translated = points_ego_expanded + trans_expanded

        # Einstein Summation 규칙을 활용한 고속 배치 행렬곱
        points_cam = torch.einsum('bcij, bcnj -> bcni', rot_ego_to_cam, points_translated)

        # ==========================================
        # [Day 20]: Intrinsic Matrix (내재 재원) 연산
        # ==========================================
        X, Y, Z = points_cam[..., 0:1], points_cam[..., 1:2], points_cam[..., 2:3]
        Z_safe = torch.clamp(Z, min=self.eps) # 0으로 나누기 방지용 안전 클리핑

        fx = intrinsics[:, :, 0, 0].unsqueeze(-1).unsqueeze(-1)
        fy = intrinsics[:, :, 1, 1].unsqueeze(-1).unsqueeze(-1)
        cx = intrinsics[:, :, 0, 2].unsqueeze(-1).unsqueeze(-1)
        cy = intrinsics[:, :, 1, 2].unsqueeze(-1).unsqueeze(-1)

        # 핀홀 카메라 투영 공식 적용
        u = (fx * X) / Z_safe + cx
        v = (fy * Y) / Z_safe + cy
        pixels_2d = torch.cat([u, v], dim=-1)

        # ==========================================
        # [Day 21]: ROI Valid Masking (유효 영역 필터) 연산
        # ==========================================
        valid_mask = self.get_valid_mask(pixels_2d, points_cam)

        return pixels_2d, valid_mask


# =========================================================
# 🛠️ 통합 데이터셋 파이프라인 최종 마감 테스트 및 디버깅 코드
# =========================================================
if __name__ == "__main__":
    # 1. 데이터셋 인스턴스 빌드 (해상도 480x640 세팅)
    dataset = NuScenesGeometryDataset(img_shape=(480, 640))

    # Batch=2, 6대 멀티 카메라, 1000개의 3D 샘플 포인트
    B, Num_Cam, N = 2, 6, 1000

    # 2. 테스트용 더미 기하학 텐서 데이터 생성
    points_ego = torch.randn(B, N, 3) * 20.0 # 넓은 전방 도로 공간 모사
    rot_ego_to_cam = torch.randn(B, Num_Cam, 3, 3)
    trans_ego_to_cam = torch.randn(B, Num_Cam, 3)

    intrinsics = torch.zeros(B, Num_Cam, 3, 3)
    for b in range(B):
        for c in range(Num_Cam):
            intrinsics[b, c] = torch.tensor([
                [700.0,   0.0, 320.0],
                [  0.0, 700.0, 240.0],
                [  0.0,   0.0,   1.0]
            ])

    # 3. 통합 파이프라인 변환 구동
    pixels, masks = dataset.pipeline_transform(points_ego, rot_ego_to_cam, trans_ego_to_cam, intrinsics)

    # 4. 셰이프 구조 및 하드웨어 정렬 검증 결과 출력
    print("=== Day 21 통합 파이프라인 최종 검증 ===")
    print("최종 투영된 픽셀 셰이프 (Pixels)   :", pixels.shape) # 결과: [2, 6, 1000, 2]
    print("유효 픽셀 필터링 마스크 셰이프 (Masks) :", masks.shape)  # 결과: [2, 6, 1000]

    # 단언문(Assertion) 차원 정렬 체크
    assert masks.shape == (B, Num_Cam, N), "Mask 차원 매칭 실패!"
    print(f"🎯 [최종 통과] 전체 3D-to-2D 투영 및 {masks.dtype} 필터 마스크 패키징이 성공적으로 마감되었습니다.")

